# Project 02: Data Visualization with Matplotlib & Seaborn

**Dataset:** Seaborn Tips sample (244 restaurant transactions).

This notebook follows the learner guide: it explores four questions, chooses appropriate charts, uses a consistent colour-blind-friendly palette, annotates the correlation heatmap, writes takeaways, and saves a 2×2 dashboard at 150 DPI.

## 1. Questions
1. How are restaurant bills distributed?
2. Do tip amounts differ by day?
3. Does a larger bill tend to come with a larger tip?
4. Which numeric variables move together?

## 2. Import libraries and load the included dataset

The CSV is included in `data/tips.csv`, so the notebook can run without downloading data.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / "data" / "tips.csv").exists():
    ROOT = Path("/content/Project_02_Data_Visualization")  # optional Colab folder

tips = pd.read_csv(ROOT / "data" / "tips.csv")
print("Dataset shape:", tips.shape)
display(tips.head())
print("\nMissing values by column:")
display(tips.isna().sum())

## 3. Style and numerical summary

In [ ]:
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
palette = sns.color_palette("colorblind")
day_order = ["Thur", "Fri", "Sat", "Sun"]

display(tips.describe(include="all"))

## 4. Histogram — distribution of total bills

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=tips, x="total_bill", bins=20, color=palette[0],
             edgecolor="white", ax=ax)
ax.set(title="Most restaurant bills are below $25",
       xlabel="Total bill (USD)", ylabel="Number of meals")
fig.tight_layout()
fig.savefig(ROOT / "charts" / "01_histogram_total_bill.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway:** Most meals have total bills below $25 (77.0% of the 244 observations).

## 5. Box plot — tips by day

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=tips, x="day", y="tip", order=day_order,
            color=palette[2], ax=ax)
ax.set(title="Tip amounts show different spreads across days",
       xlabel="Day", ylabel="Tip (USD)")
fig.tight_layout()
fig.savefig(ROOT / "charts" / "02_boxplot_tips_by_day.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway:** The median tip is highest on Sun ($3.15) in this sample.

## 6. Scatter plot — total bill versus tip

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=tips, x="total_bill", y="tip", hue="time",
                palette={"Lunch": palette[0], "Dinner": palette[1]},
                alpha=0.8, ax=ax)
ax.set(title="Larger bills generally come with larger tips",
       xlabel="Total bill (USD)", ylabel="Tip (USD)")
fig.tight_layout()
fig.savefig(ROOT / "charts" / "03_scatter_bill_vs_tip.png", dpi=150, bbox_inches="tight")
plt.show()

print("Pearson correlation:", round(tips["total_bill"].corr(tips["tip"]), 2))

**Takeaway:** Total bill and tip have a positive linear association (Pearson r = 0.68); this is association, not proof of causation.

## 7. Correlation heatmap

In [ ]:
corr = tips.select_dtypes("number").corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="crest", vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation between numeric restaurant variables")
fig.tight_layout()
fig.savefig(ROOT / "charts" / "04_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
display(corr)

**Takeaway:** The strongest numeric correlation is between total_bill and tip (r = 0.68). Correlation does not prove causation.

## 8. Bar chart — average tip by day

The vertical axis starts at zero to avoid exaggerating differences. Error bars show 95% confidence intervals.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=tips, x="day", y="tip", order=day_order,
            estimator="mean", errorbar=("ci", 95), color=palette[4], ax=ax)
ax.set(title="Average tip is highest on Sunday in this sample",
       xlabel="Day", ylabel="Average tip (USD)")
ax.set_ylim(0, tips.groupby("day")["tip"].mean().max() * 1.25)
fig.tight_layout()
fig.savefig(ROOT / "charts" / "05_bar_average_tip_by_day.png", dpi=150, bbox_inches="tight")
plt.show()

display(tips.groupby("day")["tip"].mean().sort_values(ascending=False))

**Takeaway:** Average tip is highest on Sun ($3.26) and lowest on Fri ($2.73).

## 9. Combined 2×2 dashboard (150 DPI)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
sns.histplot(data=tips, x="total_bill", bins=20, color=palette[0],
             edgecolor="white", ax=axes[0, 0])
axes[0, 0].set(title="Most bills are below $25", xlabel="Total bill (USD)",
               ylabel="Number of meals")
sns.boxplot(data=tips, x="day", y="tip", order=day_order,
            color=palette[2], ax=axes[0, 1])
axes[0, 1].set(title="Tip amounts vary across days", xlabel="Day", ylabel="Tip (USD)")
sns.scatterplot(data=tips, x="total_bill", y="tip", hue="time",
                palette={"Lunch": palette[0], "Dinner": palette[1]},
                alpha=0.8, ax=axes[1, 0])
axes[1, 0].set(title="Larger bills generally mean larger tips",
               xlabel="Total bill (USD)", ylabel="Tip (USD)")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="crest", vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=axes[1, 1])
axes[1, 1].set_title("Numeric variables are positively related")
fig.suptitle("Restaurant Tips — Data Visualization Dashboard",
             fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
fig.savefig(ROOT / "charts" / "tips_overview_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Final checklist and conclusion

- [x] Each chart type fits its question.
- [x] Every chart has a finding-based title, labelled axes and units.
- [x] Colours are consistent and colour-blind friendly.
- [x] Heatmap is annotated and explained.
- [x] A written takeaway is provided for every chart.
- [x] The combined dashboard is saved at 150 DPI.
- [x] Bar chart baseline starts at zero; layouts use `tight_layout()`.

**Conclusion:** In this sample, bill size and tip amount tend to increase together, while tips vary across days. These are descriptive patterns in this dataset and should not be generalized to all restaurants.